# Enterprise Retail Intelligence Platform

## Notebook 05 : Feature Engineering

### Author
**Shobha Saxena**

---

## Project Objective

This notebook creates additional business features from the processed datasets.

These engineered features improve analytical capabilities and will later be used for:

- Business KPI Development
- Customer Analytics
- Product Analytics
- Logistics Analysis
- Power BI Dashboard
- SQL Reporting
- Machine Learning (future scope)

---

## Output

An engineered retail dataset ready for advanced analytics and dashboard development

# Import Required Libraries

In [55]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

# Project Configuration

In [56]:
# ============================================================
# Project Configuration
# ============================================================

PROJECT_ROOT = Path("../")

DATA_PATH = PROJECT_ROOT / "data" / "processed"

OUTPUT_PATH = PROJECT_ROOT / "data" / "engineered"

# Load Processed Dataset

In [57]:
# ============================================================
# Load Required Datasets
# ============================================================

orders_df = pd.read_csv(
    DATA_PATH / "orders.csv",
    parse_dates=[
        "order_date",
        "ship_date",
        "created_at"
    ]
)

transportation_df = pd.read_csv(
    DATA_PATH / "transportation.csv",
    parse_dates=[
        "dispatch_date",
        "estimated_delivery_date",
        "actual_delivery_date",
        "created_at"
    ]
)

In [4]:
# ============================================================
# Convert Date Columns
# ============================================================

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])
orders_df["ship_date"] = pd.to_datetime(orders_df["ship_date"])

transportation_df["dispatch_date"] = pd.to_datetime(
    transportation_df["dispatch_date"]
)

transportation_df["estimated_delivery_date"] = pd.to_datetime(
    transportation_df["estimated_delivery_date"]
)

transportation_df["actual_delivery_date"] = pd.to_datetime(
    transportation_df["actual_delivery_date"]
)

print("Date Conversion Completed!")

Date Conversion Completed!


# Feature Engineering

This notebook creates new business-ready features from the processed retail data.

The engineered features improve:

- Customer Segmentation
- Sales Analysis
- Logistics Analysis
- Executive KPI Reporting
- Power BI Dashboard
- Future Machine Learning Models

In [37]:
# ============================================================
# Feature 1 : Delivery Days
# ============================================================

orders_df["delivery_days"] = (
    orders_df["ship_date"] - orders_df["order_date"]
).dt.days

orders_df[
    [
        "order_date",
        "ship_date",
        "delivery_days"
    ]
].head()

,order_date,ship_date,delivery_days
0,2017-11-08,2017-11-11,3
1,2017-11-08,2017-11-11,3
2,2017-06-12,2017-06-16,4
3,2016-10-11,2016-10-18,7
4,2016-10-11,2016-10-18,7


## Feature 2 : Order Month

In [38]:
# ============================================================
# Feature 2 : Order Month
# ============================================================

orders_df["order_month"] = (
    orders_df["order_date"]
    .dt.month_name()
)

orders_df[
    [
        "order_date",
        "order_month"
    ]
].head()

,order_date,order_month
0,2017-11-08,November
1,2017-11-08,November
2,2017-06-12,June
3,2016-10-11,October
4,2016-10-11,October


## Feature 3 : Order Quarter

In [39]:
# ============================================================
# Feature 3 : Order Quarter
# ============================================================

orders_df["order_quarter"] = (
    "Q" +
    orders_df["order_date"]
    .dt.quarter.astype(str)
)

orders_df[
    [
        "order_date",
        "order_quarter"
    ]
].head()

,order_date,order_quarter
0,2017-11-08,Q4
1,2017-11-08,Q4
2,2017-06-12,Q2
3,2016-10-11,Q4
4,2016-10-11,Q4


## Feature 4 : Order Year

In [40]:
# ============================================================
# Feature 4 : Order Year
# ============================================================

orders_df["order_year"] = (
    orders_df["order_date"]
    .dt.year
)

orders_df[
    [
        "order_date",
        "order_year"
    ]
].head()

,order_date,order_year
0,2017-11-08,2017
1,2017-11-08,2017
2,2017-06-12,2017
3,2016-10-11,2016
4,2016-10-11,2016


## Feature 5 : Weekend Order

In [41]:
# ============================================================
# Feature 5 : Weekend Order
# ============================================================

orders_df["weekend_order"] = (
    orders_df["order_date"]
    .dt.dayofweek >= 5
)

orders_df[
    [
        "order_date",
        "weekend_order"
    ]
].head()

,order_date,weekend_order
0,2017-11-08,False
1,2017-11-08,False
2,2017-06-12,False
3,2016-10-11,False
4,2016-10-11,False


## Feature 6 : High Value Order

In [42]:
# ============================================================
# Feature 6 : High Value Order
# ============================================================

HIGH_VALUE_THRESHOLD = 500

orders_df["high_value_order"] = (
    orders_df["sales"] >= HIGH_VALUE_THRESHOLD
)

orders_df[
    [
        "sales",
        "high_value_order"
    ]
].head()

,sales,high_value_order
0,261.96,False
1,731.94,True
2,14.62,False
3,957.58,True
4,22.37,False


## Feature 7 : Sales Category

In [43]:
# ============================================================
# Feature 7 : Sales Category
# ============================================================

orders_df["sales_category"] = pd.cut(
    orders_df["sales"],
    bins=[0,100,500,float("inf")],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

orders_df[
    [
        "sales",
        "sales_category"
    ]
].head()

,sales,sales_category
0,261.96,Medium
1,731.94,High
2,14.62,Low
3,957.58,High
4,22.37,Low


## Feature 8 : Customer Order Frequency

In [44]:
# ============================================================
# Feature 8 : Customer Order Frequency
# ============================================================

customer_frequency = (
    orders_df
    .groupby("customer_id")
    .size()
    .rename("order_frequency")
)

orders_df = orders_df.merge(
    customer_frequency,
    on="customer_id",
    how="left"
)

orders_df[
    [
        "customer_id",
        "order_frequency"
    ]
].head()

,customer_id,order_frequency
0,CG-12520,5
1,CG-12520,5
2,DV-13045,9
3,SO-20335,15
4,SO-20335,15


## Feature 9 : Customer Total Sales

In [45]:
# ============================================================
# Feature 9 : Customer Total Sales
# ============================================================

customer_sales = (
    orders_df
    .groupby("customer_id")["sales"]
    .sum()
    .rename("customer_total_sales")
)

orders_df = orders_df.merge(
    customer_sales,
    on="customer_id",
    how="left"
)

orders_df[
    [
        "customer_id",
        "customer_total_sales"
    ]
].head()

,customer_id,customer_total_sales
0,CG-12520,1148.78
1,CG-12520,1148.78
2,DV-13045,1119.48
3,SO-20335,2602.58
4,SO-20335,2602.58


## Feature 10 : Customer Segment

In [47]:
# ============================================================
# Feature 10 : Customer Segment
# ============================================================

orders_df["customer_segment"] = pd.qcut(
    orders_df["customer_total_sales"],
    q=3,
    labels=[
        "Low Value",
        "Medium Value",
        "High Value"
    ]
)

orders_df[
    [
        "customer_total_sales",
        "customer_segment"
    ]
].head()

,customer_total_sales,customer_segment
0,1148.78,Low Value
1,1148.78,Low Value
2,1119.48,Low Value
3,2602.58,Medium Value
4,2602.58,Medium Value


## Feature 11 : Return Flag

In [48]:
# ============================================================
# Feature 11 : Return Flag
# ============================================================

returned_orders = returns_df["order_row_id"].unique()

orders_df["return_flag"] = (
    orders_df["row_id"]
    .isin(returned_orders)
)

orders_df[
    [
        "row_id",
        "return_flag"
    ]
].head()

,row_id,return_flag
0,1,False
1,2,False
2,3,False
3,4,False
4,5,False


## Feature 12 : Transportation Efficiency

In [51]:
# ============================================================
# Convert Transportation Date Columns
# ============================================================

transportation_df["dispatch_date"] = pd.to_datetime(
    transportation_df["dispatch_date"]
)

transportation_df["estimated_delivery_date"] = pd.to_datetime(
    transportation_df["estimated_delivery_date"]
)

transportation_df["actual_delivery_date"] = pd.to_datetime(
    transportation_df["actual_delivery_date"]
)

print("Transportation dates converted successfully!")

Transportation dates converted successfully!


In [52]:
# ============================================================
# Feature 12 : Transportation Efficiency
# ============================================================

transportation_df["delivery_days"] = (
    transportation_df["actual_delivery_date"]
    - transportation_df["dispatch_date"]
).dt.days

transportation_df["transportation_efficiency"] = np.where(
    transportation_df["delivery_days"] <= 3,
    "Fast",
    np.where(
        transportation_df["delivery_days"] <= 6,
        "Normal",
        "Delayed"
    )
)

transportation_df[
    [
        "dispatch_date",
        "actual_delivery_date",
        "delivery_days",
        "transportation_efficiency"
    ]
].head()

,dispatch_date,actual_delivery_date,delivery_days,transportation_efficiency
0,2017-11-08,2017-11-12,4.0,Normal
1,2017-11-10,2017-11-13,3.0,Fast
2,2017-06-13,2017-06-18,5.0,Normal
3,2016-10-13,2016-10-18,5.0,Normal
4,2016-10-13,2016-10-16,3.0,Fast


# Save Engineered Datasets

In [53]:
# ============================================================
# Save Engineered Datasets
# ============================================================

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

orders_df.to_csv(
    OUTPUT_PATH / "orders_engineered.csv",
    index=False
)

transportation_df.to_csv(
    OUTPUT_PATH / "transportation_engineered.csv",
    index=False
)

print("✅ Engineered datasets saved successfully!")

✅ Engineered datasets saved successfully!


# Verify Engineered Features

In [54]:
# ============================================================
# Verify Engineered Features
# ============================================================

engineered_features = [
    "delivery_days",
    "order_month",
    "order_quarter",
    "order_year",
    "weekend_order",
    "high_value_order",
    "sales_category",
    "order_frequency",
    "customer_total_sales",
    "customer_segment",
    "return_flag"
]

orders_df[engineered_features].head()

,delivery_days,order_month,order_quarter,order_year,weekend_order,high_value_order,sales_category,order_frequency,customer_total_sales,customer_segment,return_flag
0,3,November,Q4,2017,False,False,Medium,5,1148.78,Low Value,False
1,3,November,Q4,2017,False,True,High,5,1148.78,Low Value,False
2,4,June,Q2,2017,False,False,Low,9,1119.48,Low Value,False
3,7,October,Q4,2016,False,True,High,15,2602.58,Medium Value,False
4,7,October,Q4,2016,False,False,Low,15,2602.58,Medium Value,False


# Final Summary

In [58]:
summary = pd.DataFrame({
    "Dataset": [
        "Orders",
        "Transportation"
    ],
    "Rows": [
        len(orders_df),
        len(transportation_df)
    ],
    "Columns": [
        orders_df.shape[1],
        transportation_df.shape[1]
    ]
})

summary

,Dataset,Rows,Columns
0,Orders,9800,14
1,Transportation,9800,10


# Conclusion

Feature Engineering successfully transformed processed retail data into business-ready analytical datasets.

## Features Created

- Delivery Days
- Order Month
- Order Quarter
- Order Year
- Weekend Order
- High Value Order
- Sales Category
- Order Frequency
- Customer Total Sales
- Customer Segment
- Return Flag
- Transportation Efficiency

## Output Files

- orders_engineered.csv
- transportation_engineered.csv

**Notebook Status:** ✅ Completed Successfully